## DSCI552 HW3

## Simon Vosskuehler

## Github: SimonVoss17

## 9916612281

In [1]:
#import packages

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## 1.(a)

## Cleaning 
bending 2/dataset4.csv: converted spaces to commas with beloew script.

cycling/dataset9.csv and cycling/dataset14.csv: removed trailing comma by hand
original files kept it AReM(original)

In [2]:
file_path = Path("ARem/bending2/dataset4.csv")
raw_lines = file_path.read_text().splitlines()

clean_lines = []
for raw_line in raw_lines:
    if raw_line.startswith('#'):
        clean_lines.append(raw_line)
    else:
        clean_lines.append(",".join(raw_line.split()))
file_path.write_text("\n".join(clean_lines) + "\n")


18630

## loading data from folder

In [3]:
raw_data = Path("AReM")
activities = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
sensor_columns = ["avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]


In [4]:
def read_instance(file_path):
    data = pd.read_csv(file_path, skiprows=5, header=None)
    data.columns = ["time"] + sensor_columns
    return data

In [5]:
all_data = {}

for activity in activities:
    file_number = len(list((raw_data / activity).glob("dataset*.csv")))
    for n in range(1, file_number +1):
        all_data[(activity, n)] = read_instance(raw_data / activity / f"dataset{n}.csv")

len(all_data)

88

## 1.(b)

In [6]:
test_ids = {
    "bending1": [1,2],
    "bending2": [1,2],
    "cycling": [1,2,3],
    "lying": [1,2,3],
    "sitting": [1,2,3],
    "standing":[1,2,3],
    "walking":[1,2,3]
}

train_data = {}
test_data = {}

In [7]:
for key, data in all_data.items():
    act, num = key
    if num in test_ids[act]:
        test_data[key] = data
    else:
        train_data[key] = data

In [8]:
print(len(train_data), len(test_data))

69 19


69 datasets in train, 19 datasets in test

## 1.(c)i.

Mean

Min

Max

Range

Variance

Standard deviation

Root mean square

interquartile range



## 1.(c)ii.

In [9]:
# get summary stats for an instance and turn into pandas series

def get_features(data):
    signals = data[sensor_columns]
    summary = pd.DataFrame({
        "min": signals.min(),
        "max": signals.max(),
        "mean": signals.mean(),
        "median": signals.median(),
        "std": signals.std(),
        "1st_quartile": signals.quantile(0.25),
        "3nd_quartile": signals.quantile(0.75)
    })
    summary.index = range(1,7)
    row = summary.T.unstack()
    row.index = [f"{stat}{i}" for i, stat in row.index]
    return row


In [10]:
# build feature table

feature_df = pd.DataFrame([get_features(data) for data in all_data.values()])
feature_df.index = range(1, len(feature_df)+1)
feature_df.index.name = "Instance"
feature_df.shape


(88, 42)

In [11]:
feature_df

,min1,max1,mean1,median1,std1,1st_quartile1,3nd_quartile1,min2,max2,mean2,...,std5,1st_quartile5,3nd_quartile5,min6,max6,mean6,median6,std6,1st_quartile6,3nd_quartile6
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,4.576562,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,4.456333,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,4.371958,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350


## 1.(c)iii.

In [12]:
# get standard deviations

feature_df.std()

min1             9.569975
max1             4.394362
mean1            5.335718
median1          5.440054
std1             1.772153
1st_quartile1    6.153590
3nd_quartile1    5.138925
min2             0.000000
max2             5.062729
mean2            1.574164
median2          1.412244
std2             0.884105
1st_quartile2    0.946386
3nd_quartile2    2.125266
min3             2.956462
max3             4.875137
mean3            4.008380
median3          4.036396
std3             0.946710
1st_quartile3    4.220658
3nd_quartile3    4.171628
min4             0.000000
max4             2.183625
mean4            1.166114
median4          1.145586
std4             0.458242
1st_quartile4    0.843620
3nd_quartile4    1.552504
min5             6.124001
max5             5.741238
mean5            5.675593
median5          5.813782
std5             1.024898
1st_quartile5    6.096465
3nd_quartile5    5.531720
min6             0.045838
max6             2.518921
mean6            1.154812
median6     

In [13]:
# bootstrap
N = 1000
bootstrap_stds = []
for n in range(N):
    sample = feature_df.sample(n=len(feature_df), replace = True, random_state=1234)
    bootstrap_stds.append(sample.std())

bootstrapped_stds = pd.DataFrame(bootstrap_stds)
bootstrapped_stds




,min1,max1,mean1,median1,std1,1st_quartile1,3nd_quartile1,min2,max2,mean2,...,std5,1st_quartile5,3nd_quartile5,min6,max6,mean6,median6,std6,1st_quartile6,3nd_quartile6
0,9.761751,4.382088,5.409809,5.498589,1.816925,6.301975,5.046454,0.0,5.539216,1.65927,...,1.144133,6.019732,5.471561,0.045838,2.526415,1.191403,1.115318,0.537776,0.773396,1.574158
1,9.761751,4.382088,5.409809,5.498589,1.816925,6.301975,5.046454,0.0,5.539216,1.65927,...,1.144133,6.019732,5.471561,0.045838,2.526415,1.191403,1.115318,0.537776,0.773396,1.574158
2,9.761751,4.382088,5.409809,5.498589,1.816925,6.301975,5.046454,0.0,5.539216,1.65927,...,1.144133,6.019732,5.471561,0.045838,2.526415,1.191403,1.115318,0.537776,0.773396,1.574158
3,9.761751,4.382088,5.409809,5.498589,1.816925,6.301975,5.046454,0.0,5.539216,1.65927,...,1.144133,6.019732,5.471561,0.045838,2.526415,1.191403,1.115318,0.537776,0.773396,1.574158
4,9.761751,4.382088,5.409809,5.498589,1.816925,6.301975,5.046454,0.0,5.539216,1.65927,...,1.144133,6.019732,5.471561,0.045838,2.526415,1.191403,1.115318,0.537776,0.773396,1.574158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,9.761751,4.382088,5.409809,5.498589,1.816925,6.301975,5.046454,0.0,5.539216,1.65927,...,1.144133,6.019732,5.471561,0.045838,2.526415,1.191403,1.115318,0.537776,0.773396,1.574158
996,9.761751,4.382088,5.409809,5.498589,1.816925,6.301975,5.046454,0.0,5.539216,1.65927,...,1.144133,6.019732,5.471561,0.045838,2.526415,1.191403,1.115318,0.537776,0.773396,1.574158
997,9.761751,4.382088,5.409809,5.498589,1.816925,6.301975,5.046454,0.0,5.539216,1.65927,...,1.144133,6.019732,5.471561,0.045838,2.526415,1.191403,1.115318,0.537776,0.773396,1.574158
998,9.761751,4.382088,5.409809,5.498589,1.816925,6.301975,5.046454,0.0,5.539216,1.65927,...,1.144133,6.019732,5.471561,0.045838,2.526415,1.191403,1.115318,0.537776,0.773396,1.574158


In [14]:
std_ci = pd.DataFrame({
    "std": feature_df.std(),
    "CI_lower": bootstrapped_stds.quantile(0.05),
    "CI_upper": bootstrapped_stds.quantile(0.95),
})
std_ci.round(3)

,std,CI_lower,CI_upper
min1,9.570,9.762,9.762
max1,4.394,4.382,4.382
mean1,5.336,5.410,5.410
median1,5.440,5.499,5.499
std1,1.772,1.817,1.817
1st_quartile1,6.154,6.302,6.302
3nd_quartile1,5.139,5.046,5.046
min2,0.000,0.000,0.000
max2,5.063,5.539,5.539
mean2,1.574,1.659,1.659


## 1.(c)iv.

Selected features:

Mean: captures average signal strength between two sensors which varies depending on activity.

Standard Deviation: captures how much the signals vary, static activities will vary less than moving ones

Maximum: Captures spikes from sudden movements

## 2 ISLR 3.7.4

(a) On the train set the cubic will be lower because it will fit to some amount of random noise.

(b) On the test set the linear will be lower because the cubic is overfit to noise in the train set

(c) For the same reason as (a), cubic will be lower on the train set. The cubic also contains the liner, when beta2 and beta3 are zero, so it can never be worse.

(d) There is not enough information. If the relationship is very close to linear then it the linear will before better. If the relationship is very nonlinear then the cubic will peform better.